Referência
https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/agents/gemini_data_analytics/intro_gemini_data_analytics_http.ipynb

### Var e Import

In [20]:
import google
import requests
import json


In [38]:
billing_project = "llm-studies"
location = "global"
api_version = "v1beta"
dataset_id = "analytics_527405664"
base_url = "https://geminidataanalytics.googleapis.com"
data_agent_url = f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}/dataAgents"
data_agent_url

'https://geminidataanalytics.googleapis.com/v1beta/projects/llm-studies/locations/global/dataAgents'

'https://geminidataanalytics.googleapis.com/v1beta/projects/llm-studies/locations/global/dataAgents'

### Auth - revisar!

In [9]:
google.auth.default()

access_token = !gcloud auth application-default print-access-token
headers = {
    "Authorization": f"Bearer {access_token[0]}",
    "Content-Type": "application/json",
}

In [ ]:
import google.auth
import google.auth.transport.requests

credentials, project = google.auth.default()
auth_request = google.auth.transport.requests.Request()
credentials.refresh(auth_request)

headers = {
    "Authorization": f"Bearer {credentials.token}",
    "Content-Type": "application/json"
}

### Agent

In [50]:
data_agent_id = "ga_data_agent"

In [ ]:
datasource_references = {
    "bq": {
        "tableReferences": [
            {
                "projectId": "llm-studies",
                "datasetId": "analytics_527405664",
                "tableId": "events_20260306",
            }
        ]
    }
}

In [27]:
example_queries = [
    {
        "naturalLanguageQuestion": "Conte quantas linhas existem na tabela de eventos do dia 10 de março de 2026",
        "sqlQuery": "SELECT count(*) as total_eventos FROM  events_20260306",
    },
    {
        "naturalLanguageQuestion": "Quantos IDs de usuários diferentes (únicos) interagiram com o app hoje?",
        "sqlQuery": "SELECT   count(DISTINCT user_pseudo_id) as usuarios_unicos FROM events_20260306",
    },
]

glossary_terms = [
    {
        "display_name": "Event Parameters",
        "description": "Um campo do tipo 'record' repetido que armazena detalhes extras sobre uma ação do usuário, como o nome de um botão ou o valor de uma compra.",
        "labels": ["nested", "metadata", "key-value"]
    },
    {
        "display_name": "User Pseudo ID",
        "description": "Identificador único e anônimo gerado pelo Firebase para cada instância de instalação do app ou navegador, permitindo rastrear o comportamento sem identificar a pessoa.",
        "labels": ["analytics", "identity", "privacy"]
    },
    {
        "display_name": "UNNEST",
        "description": "Função SQL do BigQuery usada para 'achatar' (flatten) campos repetidos, transformando os elementos de um array em linhas individuais para consulta.",
        "labels": ["sql", "array", "flattening"]
    },
    {
        "display_name": "Table Suffix",
        "description": "Operador coringa (_TABLE_SUFFIX) usado para filtrar e consultar múltiplas tabelas particionadas por data simultaneamente.",
        "labels": ["wildcard", "filtering", "partition"]
    },
    {
        "display_name": "Event Timestamp",
        "description": "O momento exato em que o evento ocorreu, registrado em microssegundos desde o início da Era Unix (UTC).",
        "labels": ["time", "epoch", "precision"]
    },
    {
        "display_name": "User Properties",
        "description": "Atributos persistentes que descrevem segmentos da sua base de usuários, como preferência de idioma ou status de assinatura.",
        "labels": ["user", "segmentation", "nested"]
    },
    {
        "display_name": "Stream ID",
        "description": "O identificador do fluxo de dados (iOS, Android ou Web) de onde o evento se originou.",
        "labels": ["source", "platform", "filter"]
    }
]
use_glossary_terms = True
use_example_queries = True

In [24]:
system_instruction = "Pense como um analista de negócio e marketing que entender o comportamento do usuário da aplicação"

In [28]:
data_agent_payload = {
    "name": f"projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}",  # Optional
    "description": "Data agent to analyze firebase events.",
    "data_analytics_agent": {
        "published_context": {
            "datasource_references": datasource_references,
            "system_instruction": system_instruction,
            "options": {
                "analysis": {
                    # Optional - if wanting to use advanced analysis with python.
                    # Default is False.
                    "python": {"enabled": False}
                }
            },
        }
    },
}

data_agent_payload["data_analytics_agent"]["published_context"]["example_queries"] = example_queries
data_agent_payload["data_analytics_agent"]["published_context"]["glossary_terms"] = glossary_terms
params = {"data_agent_id": data_agent_id}

In [29]:
data_agent_response = requests.post(
    data_agent_url, params=params, json=data_agent_payload, headers=headers
)
print(json.dumps(data_agent_response.json(), indent=2))

{
  "name": "projects/llm-studies/locations/global/operations/operation-1773229252557-64cbe1bb43a20-16a01f92-29372762",
  "metadata": {
    "@type": "type.googleapis.com/google.cloud.geminidataanalytics.v1beta.OperationMetadata",
    "createTime": "2026-03-11T11:40:52.816860149Z",
    "target": "projects/llm-studies/locations/global/dataAgents/ga_data_agent",
    "verb": "create",
    "requestedCancellation": false,
    "apiVersion": "v1beta"
  },
  "done": false
}


In [51]:
data_agent_url = f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}"

data_agent_response = requests.get(data_agent_url, headers=headers)

if data_agent_response.status_code == 200:
    print("Fetched Data Agent successfully!")
    print(json.dumps(data_agent_response.json(), indent=2))
else:
    print(f"Error: {data_agent_response.status_code}")
    print(data_agent_response.text)

Fetched Data Agent successfully!
{
  "name": "projects/llm-studies/locations/global/dataAgents/ga_data_agent",
  "description": "Data agent to analyze firebase events.",
  "createTime": "2026-03-11T11:40:52.568440528Z",
  "updateTime": "2026-03-11T11:40:53.743338335Z",
  "dataAnalyticsAgent": {
    "publishedContext": {
      "systemInstruction": "Pense como um analista de neg\u00f3cio e marketing que entender o comportamento do usu\u00e1rio da aplica\u00e7\u00e3o",
      "options": {
        "analysis": {
          "python": {}
        }
      },
      "exampleQueries": [
        {
          "naturalLanguageQuestion": "Conte quantas linhas existem na tabela de eventos do dia 10 de mar\u00e7o de 2026",
          "sqlQuery": "SELECT count(*) as total_eventos FROM  events_20260306"
        },
        {
          "naturalLanguageQuestion": "Quantos IDs de usu\u00e1rios diferentes (\u00fanicos) interagiram com o app hoje?",
          "sqlQuery": "SELECT   count(DISTINCT user_pseudo_id) as 

In [ ]:
new_datasource = {
    "bigquery_reference": {
        "table_id": "llm-studies.analytics_527405664.events_*",
    }
}

In [ ]:
new_datasource = {
    "bq": {
        "tableReferences": [
            {
                "projectId": "llm-studies",
                "datasetId": "analytics_527405664",
                "tableId": "events_20260306",
            }
        ]
    }
}

In [33]:
# Certifique-se de que ga_data_agent_id seja o caminho completo:
# projects/llm-studies/locations/global/dataAgents/ga_data_agent

full_resource_name = f"projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}"
update_url = f"https://geminidataanalytics.googleapis.com/v1beta/{full_resource_name}"

In [35]:
updated_datasources = [
    {"bigquery_reference": {"table_id": "llm-studies.analytics_527405664.events_*"}},
    {"bigquery_reference": {"table_id": "llm-studies.analytics_527405664.events_intraday_*"}} # Nova tabela com wildcard
]

In [52]:
update_payload = {
    "data_analytics_agent": {
        "published_context": {
            "datasourceReferences": [  # O nome correto é data_sources
                {
                    "bigquery_table": {
                        # O formato deve ser o caminho total da tabela ou wildcard
                        "table_uri": f"projects/{billing_project}/datasets/{dataset_id}/tables/events_*"
                    }
                }
            ]
        }
    }
}

# No updateMask, você também deve refletir o nome correto do campo
update_params = {
    "updateMask": "data_analytics_agent.published_context.datasourceReferences"
}

In [54]:
new_table_references = [
    {
        "projectId": billing_project,
        "datasetId": "analytics_527405664",
        "tableId": "events_*"  # O wildcard aqui
    },
    {
        "projectId": billing_project,
        "datasetId": "analytics_527405664",
        "tableId": "events_intraday_*"  # O wildcard aqui
    }
]

In [55]:
update_payload = {
    "dataAnalyticsAgent": {
        "publishedContext": {
            "datasourceReferences": {
                "bq": {
                    "tableReferences": new_table_references
                }
            }
        }
    }
}

# O updateMask segue o caminho do campo usando a nomenclatura do backend
update_params = {
    "updateMask": "data_analytics_agent.published_context.datasource_references"
}

In [56]:

response = requests.patch(
    update_url, 
    params=update_params, 
    json=update_payload, 
    headers=headers
)

print(json.dumps(response.json(), indent=2))

{
  "name": "projects/llm-studies/locations/global/operations/operation-1773230572937-64cbe6a67a09a-0ca24f35-598eb59c",
  "metadata": {
    "@type": "type.googleapis.com/google.cloud.geminidataanalytics.v1beta.OperationMetadata",
    "createTime": "2026-03-11T12:02:53.116881123Z",
    "target": "projects/llm-studies/locations/global/dataAgents/ga_data_agent",
    "verb": "update",
    "requestedCancellation": false,
    "apiVersion": "v1beta"
  },
  "done": false
}


In [57]:
new_example_queries = [
    {
        "naturalLanguageQuestion": "Quantos eventos de 'session_start' tivemos nos últimos 7 dias?",
        "sqlQuery": f"SELECT count(*) as total_sessoes FROM `{billing_project}.analytics_527405664.events_*` WHERE event_name = 'session_start' AND _TABLE_SUFFIX BETWEEN FORMAT_DATE('%Y%m%d', DATE_SUB(CURRENT_DATE(), INTERVAL 7 DAY)) AND FORMAT_DATE('%Y%m%d', DATE_SUB(CURRENT_DATE(), INTERVAL 1 DAY))"
    },
    {
        "naturalLanguageQuestion": "Qual o top 5 países com mais usuários ativos ontem?",
        "sqlQuery": f"SELECT geo.country, count(DISTINCT user_pseudo_id) as usuarios FROM `{billing_project}.analytics_527405664.events_*` WHERE _TABLE_SUFFIX = FORMAT_DATE('%Y%m%d', DATE_SUB(CURRENT_DATE(), INTERVAL 1 DAY)) GROUP BY 1 ORDER BY 2 DESC LIMIT 5"
    },
    {
        "naturalLanguageQuestion": "Busque os parâmetros do evento 'page_view' para entender o fluxo de navegação de hoje.",
        "sqlQuery": f"SELECT event_name, (SELECT value.string_value FROM UNNEST(event_params) WHERE key = 'page_location') as page_path FROM `{billing_project}.analytics_527405664.events_*` WHERE event_name = 'page_view' AND _TABLE_SUFFIX = FORMAT_DATE('%Y%m%d', CURRENT_DATE())"
    }
]

In [58]:
update_payload = {
    "dataAnalyticsAgent": {
        "publishedContext": {
            "exampleQueries": new_example_queries
        }
    }
}

# Importante: O mask deve usar o padrão snake_case da API interna
update_params = {
    "updateMask": "data_analytics_agent.published_context.example_queries"
}

response = requests.patch(
    data_agent_url, 
    params=update_params, 
    json=update_payload, 
    headers=headers
)

if response.status_code == 200:
    print("Queries de exemplo atualizadas com sucesso!")
else:
    print(f"Erro {response.status_code}: {response.text}")

Queries de exemplo atualizadas com sucesso!


In [59]:
# Adaptando os nomes dos campos para o padrão esperado pela API (camelCase)
formatted_glossary = [
    {
        "displayName": term["display_name"],
        "description": term["description"],
        "labels": term["labels"]
    } for term in glossary_terms
]

# Exemplo de queries usando o novo formato de wildcard que configuramos
formatted_examples = [
    {
        "naturalLanguageQuestion": "Quantos eventos tivemos no total nos últimos 3 dias?",
        "sqlQuery": f"SELECT count(*) as total FROM `{billing_project}.analytics_527405664.events_*` WHERE _TABLE_SUFFIX BETWEEN FORMAT_DATE('%Y%m%d', DATE_SUB(CURRENT_DATE(), INTERVAL 3 DAY)) AND FORMAT_DATE('%Y%m%d', CURRENT_DATE())"
    },
    {
        "naturalLanguageQuestion": "Quantos usuários únicos interagiram com o app hoje (incluindo dados em tempo real)?",
        "sqlQuery": f"SELECT count(DISTINCT user_pseudo_id) as usuarios FROM `{billing_project}.analytics_527405664.events_intraday_*` WHERE _TABLE_SUFFIX = FORMAT_DATE('%Y%m%d', CURRENT_DATE())"
    }
]

update_payload = {
    "dataAnalyticsAgent": {
        "publishedContext": {
            "exampleQueries": formatted_examples,
            "glossaryTerms": formatted_glossary
        }
    }
}

In [60]:
# O mask usa snake_case para os nomes dos campos internos do protocolo
update_params = {
    "updateMask": "data_analytics_agent.published_context.example_queries,data_analytics_agent.published_context.glossary_terms"
}

response = requests.patch(
    data_agent_url, 
    params=update_params, 
    json=update_payload, 
    headers=headers
)

if response.status_code == 200:
    print("Glossário e Exemplos atualizados com sucesso!")
    # Opcional: imprimir o JSON para confirmar as mudanças
    # print(json.dumps(response.json(), indent=2))
else:
    print(f"Erro {response.status_code}: {response.text}")

Glossário e Exemplos atualizados com sucesso!


In [22]:
data_agent_id = "data_agent_1"  # @param {type:"string"}
conversation_id = "conversation_1"  # @param {type:"string"}
conversation_url = f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}/conversations"
conversation_payload = {
    "agents": [
        f"projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}"
    ],
    "name": f"projects/{billing_project}/locations/{location}/conversations/{conversation_id}",
}
params = {"conversation_id": conversation_id}

conversation_response = requests.post(
    conversation_url, headers=headers, params=params, json=conversation_payload
)

if conversation_response.status_code == 200:
    print("Conversation created successfully!")
    print(json.dumps(conversation_response.json(), indent=2))
else:
    print(f"Error creating Conversation: {conversation_response.status_code}")
    print(conversation_response.text)

Conversation created successfully!
{
  "name": "projects/llm-studies/locations/global/conversations/conversation_1",
  "agents": [
    "projects/llm-studies/locations/global/dataAgents/data_agent_1"
  ],
  "createTime": "2026-03-09T16:25:46.528536Z",
  "lastUsedTime": "2026-03-09T16:25:47.051255Z"
}


In [23]:
conversation_id = "conversation_1"  # @param {type:"string"}
conversation_url = f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}/conversations/{conversation_id}"
conversation_response = requests.get(conversation_url, headers=headers)
# Handle the response
if conversation_response.status_code == 200:
    print("Conversation fetched successfully!")
    print(json.dumps(conversation_response.json(), indent=2))
else:
    print(f"Error while fetching conversation: {conversation_response.status_code}")
    print(conversation_response.text)

Conversation fetched successfully!
{
  "name": "projects/llm-studies/locations/global/conversations/conversation_1",
  "agents": [
    "projects/llm-studies/locations/global/dataAgents/data_agent_1"
  ],
  "createTime": "2026-03-09T16:25:46.528536Z",
  "lastUsedTime": "2026-03-09T16:25:47.051255Z"
}


In [24]:
conversation_url = f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}/conversations"


conversation_response = requests.get(conversation_url, headers=headers)

# Handle the response
if conversation_response.status_code == 200:
    print("Conversation fetched successfully!")
    print(json.dumps(conversation_response.json(), indent=2))
else:
    print(f"Error while fetching conversation: {conversation_response.status_code}")
    print(conversation_response.text)

Conversation fetched successfully!
{
  "conversations": [
    {
      "name": "projects/llm-studies/locations/global/conversations/conversation_1",
      "agents": [
        "projects/llm-studies/locations/global/dataAgents/data_agent_1"
      ],
      "createTime": "2026-03-09T16:25:46.528536Z",
      "lastUsedTime": "2026-03-09T16:25:47.051255Z"
    }
  ]
}


In [25]:
# @title Streaming Chat Messages

def is_json(str):
    try:
        json_object = json_lib.loads(str)
    except ValueError:
        return False
    return True

def handle_text_response(resp):
    parts = resp["parts"]
    full_text = "".join(parts)
    if "\n" not in full_text and len(full_text) > 80:
        wrapped_text = textwrap.fill(full_text, width=80)
        print(wrapped_text)
    else:
        print(full_text)

def get_property(data, field_name, default=""):
    return data[field_name] if field_name in data else default

def display_schema(data):
    fields = data["fields"]
    df = pd.DataFrame(
        {
            "Column": map(lambda field: get_property(field, "name"), fields),
            "Type": map(lambda field: get_property(field, "type"), fields),
            "Description": map(
                lambda field: get_property(field, "description", "-"), fields
            ),
            "Mode": map(lambda field: get_property(field, "mode"), fields),
        }
    )
    display(df)

def display_section_title(text):
    display(HTML(f"<h2>{text}</h2>"))

def format_bq_table_ref(table_ref):
    return "{}.{}.{}".format(
        table_ref["projectId"], table_ref["datasetId"], table_ref["tableId"]
    )

def format_looker_table_ref(table_ref):
    return "lookmlModel: {}, explore: {}".format(
        table_ref["lookmlModel"], table_ref["explore"]
    )

def display_datasource(datasource):
    source_name = ""

    if "studioDatasourceId" in datasource:
        source_name = datasource["studioDatasourceId"]
    elif "lookerExploreReference" in datasource:
        source_name = format_looker_table_ref(datasource["lookerExploreReference"])
    else:
        source_name = format_bq_table_ref(datasource["bigqueryTableReference"])

    print(source_name)
    if "schema" in datasource:
        display_schema(datasource["schema"])

def handle_schema_response(resp):
    if "query" in resp:
        print(resp["query"]["question"])
    elif "result" in resp:
        display_section_title("Schema resolved")
        print("Data sources:")
        for datasource in resp["result"]["datasources"]:
            display_datasource(datasource)

def handle_data_response(resp):
    if "query" in resp:
        query = resp["query"]
        display_section_title("Retrieval query")
        if "name" in query:
            print("Query name: {}".format(query["name"]))
        if "question" in query:
          print("Question: {}".format(query["question"]))
        if "datasources" in query:
          print("Data sources:")
          for datasource in query["datasources"]:
              display_datasource(datasource)
    elif "generatedSql" in resp:
        display_section_title("SQL generated")
        print(resp["generatedSql"])
    elif "result" in resp:
        display_section_title("Data retrieved")

        fields = map(
            lambda field: get_property(field, "name"),
            resp["result"]["schema"]["fields"],
        )
        dict = {}

        for field in fields:
            dict[field] = [get_property(el, field) for el in resp["result"]["data"]]

        display(pd.DataFrame(dict))

def handle_chart_response(resp):
    if "query" in resp:
        print(resp["query"]["instructions"])
    elif "result" in resp:
        vegaConfig = resp["result"]["vegaConfig"]
        alt.Chart.from_json(json_lib.dumps(vegaConfig)).display()

def handle_clarification_response(resp):
    display_section_title("Clarification required")

    clarification_questions = ""
    for q in resp["questions"]:
        # Add the question text
        clarification_questions += f"{q['question']}\n"
        # Add the options as bullet points
        for option in q["options"]:
            clarification_questions += f"* {option}\n"
        clarification_questions += "\n" # Add a newline between questions
    print(clarification_questions)

def handle_error(resp):
    display_section_title("Error")
    print("Code: {}".format(resp["code"]))
    print("Message: {}".format(resp["message"]))

def get_stream(url, json):
    s = requests.Session()

    acc = ""

    with s.post(url, json=json, headers=headers, stream=True) as resp:
        for line in resp.iter_lines():
            if not line:
                continue

            decoded_line = str(line, encoding="utf-8")

            if decoded_line == "[{":
                acc = "{"
            elif decoded_line == "}]":
                acc += "}"
            elif decoded_line == ",":
                continue
            else:
                acc += decoded_line

            if not is_json(acc):
                continue

            data_json = json_lib.loads(acc)

            if "systemMessage" not in data_json:
                if "error" in data_json:
                    handle_error(data_json["error"])
                continue

            if "text" in data_json["systemMessage"]:
                handle_text_response(data_json["systemMessage"]["text"])
            elif "schema" in data_json["systemMessage"]:
                handle_schema_response(data_json["systemMessage"]["schema"])
            elif "data" in data_json["systemMessage"]:
                handle_data_response(data_json["systemMessage"]["data"])
            elif "chart" in data_json["systemMessage"]:
                handle_chart_response(data_json["systemMessage"]["chart"])
            elif "clarification" in data_json["systemMessage"]:
                handle_clarification_response(data_json["systemMessage"]["clarification"])
            else:
                colored_json = highlight(
                    acc, lexers.JsonLexer(), formatters.TerminalFormatter()
                )
                print(colored_json)
            print("\n")
            acc = ""

In [26]:
data_agent_id = "data_agent_1"      # @param {type:"string"}
conversation_id = "conversation_1"  # @param {type:"string"}

# fmt: off
question = "Make a bar graph for the top 5 states by the total number of airports"  # @param {type:"string"}
# fmt: on

chat_url = (
    f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}:chat"
)

 # Construct the payload
chat_payload = {
    "parent": f"projects/{billing_project}/locations/global",
    "messages": [{"userMessage": {"text": question}}],
    "conversation_reference": {
        "conversation": f"projects/{billing_project}/locations/{location}/conversations/{conversation_id}",
        "data_agent_context": {
            "data_agent": f"projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}",
        },
    },
}

if looker_selected:
    chat_payload["conversation_reference"]["data_agent_context"]["credentials"] = (
        looker_credentials
    )

# Call get_stream method to stream the response
get_stream(chat_url, chat_payload)

Retrieved contextRetrieved context for 2 tables.


Constructing the SQL QueryI've successfully identified the table and drafted the
SQL query. The key is now generating the Python code to visualize the resulting
data as a bar graph, using the `google:python_interpreter`. I am ready to move
to the next step.


Analyzing Airport DataI've zeroed in on the top 5 states by airport count. Now,
I am structuring the data. I'm building a plan to use Altair to make a bar chart
and identifying data names, chart IDs, and column details. Encoding will be
handled by specifying 'X' and 'Y' for the appropriate columns. The goal is to
produce the final answer, a bar chart of the data.


Considering Airport DataI've reviewed the data and the chart is ready to go. My
focus is on the top five states by airport count. Currently, I see that Texas is
leading by a large margin, followed by Florida, California, Alaska, and
Pennsylvania. I'm wondering why Texas has such a significant lead.


### Summary
This fi

Data sources:
bigquery-public-data.faa.us_airports




SELECT
    airports.state_abbreviation,
    COUNT(*) AS airport_count
FROM
    `bigquery-public-data.faa.us_airports` AS airports
GROUP BY
    airports.state_abbreviation
ORDER BY
    airport_count DESC
LIMIT 5;






,state_abbreviation,airport_count
0,TX,2033
1,FL,886
2,CA,870
3,AK,762
4,PA,759


Data sources:
bigquery-public-data.faa.us_airports




SELECT
    airports.state_abbreviation,
    COUNT(*) AS airport_count
FROM
    `bigquery-public-data.faa.us_airports` AS airports
GROUP BY
    airports.state_abbreviation
ORDER BY
    airport_count DESC
LIMIT 5;






KeyError: 'instructions'